# Atividade 8 — Entrega final
**Spam Blocker · Lucas da Silva Vargas · 24/09/2026**

Preparação de SMS para futura classificação entre mensagens legítimas e spam. Esta versão consolida o esboço e o feedback: adiciona normalização de espaços, reparo restrito de pontuação, auditoria da leitura, README e registro de decisões exportado.

**Fonte:** Almeida, T. & Hidalgo, J. (2011). SMS Spam Collection. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84. Licença da base: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), conforme a [UCI](https://archive.ics.uci.edu/dataset/228/sms+spam+collection). A versão tratada altera espaços e cinco caracteres de pontuação, remove duplicatas e codifica o alvo; não altera o original.

**Reprodução:** instale pandas, mantenha `sms_original.zip` ao lado do notebook e execute todas as células na ordem. Se o ZIP não estiver presente, a primeira célula baixa a fonte. No Colab, envie o ZIP ou permita o download. Salve o notebook com as saídas. O script `aula8.py` contém as mesmas células de código.

In [1]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import zipfile
import io
import csv
import pandas as pd

origem = Path("sms_original.zip")
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
if not origem.exists():
    with urlopen(url, timeout=60) as resposta:
        origem.write_bytes(resposta.read())
conteudo = origem.read_bytes()
hash_original = hashlib.sha256(conteudo).hexdigest()
with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
    texto = pacote.read("SMSSpamCollection").decode("utf-8", errors="strict")
print("SHA-256 do original:", hash_original)
print("Pandas:", pd.__version__)


SHA-256 do original: 1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3
Pandas: 2.2.3


## 1. Leitura canônica e auditoria
A Atividade 5 identificou 50 mensagens fragmentadas nas colunas `Unnamed` do CSV derivado. Consultar o TSV original evita reconstruções incertas e mantém `spam.csv` intacto.

A revisão identificou outro detalhe: o carregamento padrão interpreta aspas literais de SMS como delimitadores e une três mensagens em uma linha. Uso `quoting=csv.QUOTE_NONE`, pois cada registro da fonte é uma linha com rótulo e texto separados por tabulação, sem regra de escape por aspas. Confiro a leitura contra uma separação independente da primeira tabulação de cada linha. Assim, o início passa de **5.572 no esboço para 5.574 registros**, sem inventar mensagens.

Não retiro aspas nem linhas do original. Uma mudança futura no formato fará a conferência falhar, exigindo inspeção.

In [2]:
leitura_esboco = pd.read_csv(io.StringIO(texto), sep="\t", header=None,
                             names=["label", "message"])
df = pd.read_csv(io.StringIO(texto), sep="\t", header=None,
                 names=["label", "message"], quoting=csv.QUOTE_NONE)
linhas_fonte = texto.splitlines()
assert all(linha.startswith(("ham\t", "spam\t")) for linha in linhas_fonte)
referencia = pd.DataFrame([linha.split("\t", 1) for linha in linhas_fonte],
                         columns=["label", "message"])
# A comparação garante que cada SMS foi carregado sem unir registros.
pd.testing.assert_frame_equal(df, referencia, check_dtype=False)
df_original = df.copy(deep=True)
print("Leitura do esboço:", leitura_esboco.shape)
print("Leitura final:", df.shape)
print("Linhas físicas na fonte:", len(linhas_fonte))
print("Registros com quebra de linha na leitura anterior:",
      int(leitura_esboco["message"].str.contains("\n").sum()))
print("Tipos:\n" + df.dtypes.to_string())
print("Ausentes:\n" + df.isna().sum().to_string())
print("Duplicatas exatas:", int(df.duplicated().sum()))
print("Classes iniciais:\n" + pd.DataFrame({
    "quantidade": df["label"].value_counts(),
    "percentual": df["label"].value_counts(normalize=True).mul(100).round(2)
}).to_string())
assert set(df["label"].unique()) == {"ham", "spam"}
assert df.isna().sum().sum() == 0
assert df["message"].str.strip().ne("").all()
decisoes = []
def registrar(acao, coluna, motivo, impacto, risco):
    decisoes.append({"transformacao": acao, "coluna_afetada": coluna,
                     "motivo": motivo, "impacto": impacto, "risco": risco})
registrar("Leitura TSV sem interpretação de aspas", "label e message",
          "Preservar cada linha canônica; evitar reconstruir o CSV fragmentado",
          f"Leitura: {len(leitura_esboco)} -> {len(df)} linhas; 2 colunas; original intacto",
          "Mudança futura de formato; comparação linha a linha valida esta leitura")


Leitura do esboço: (5572, 2)
Leitura final: (5574, 2)
Linhas físicas na fonte: 5574
Registros com quebra de linha na leitura anterior: 1
Tipos:
label      object
message    object
Ausentes:
label      0
message    0
Duplicatas exatas: 403
Classes iniciais:
       quantidade  percentual
label                        
ham          4827        86.6
spam          747        13.4


## 2. Deduplicação inicial
Mantenho a primeira ocorrência de cada par `label`/`message`. Conservar cópias daria peso adicional à mesma mensagem e poderia colocar textos iguais em treino e teste. O risco é alterar frequências reais e proporções de classes; por isso, registro as contagens antes e depois. A regra não identifica textos apenas semelhantes.

In [3]:
antes = len(df)
duplicatas_iniciais = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Deduplicação inicial:", antes, "->", len(df))
print("Removidas:", duplicatas_iniciais)
registrar("Deduplicação inicial", "label e message", "Evitar peso adicional de cópias exatas",
          f"{duplicatas_iniciais} linhas removidas; {antes} -> {len(df)}; 0 colunas removidas",
          "Altera frequências; mensagens semelhantes ainda podem existir")


Deduplicação inicial: 5574 -> 5171
Removidas: 403


## 3. Normalização de espaços
Substituo sequências de espaços em branco por um espaço e retiro espaços nas extremidades. Isso reduz variações de formatação sem apagar palavras, números, URLs, maiúsculas ou pontuação. Prefiro essa regra limitada a apagar todos os espaços, o que juntaria palavras. O risco é perder informação de estilo, como a quantidade de espaços usada pelo remetente.

In [4]:
normalizado = df["message"].str.replace(r"\s+", " ", regex=True).str.strip()
linhas_espacos = int(normalizado.ne(df["message"]).sum())
df["message"] = normalizado
assert df["message"].ne("").all()
print("Mensagens com espaços normalizados:", linhas_espacos)
print("Duplicatas após espaços:", int(df.duplicated().sum()))
registrar("Normalização de espaços", "message", "Reduzir variações de formatação preservando palavras",
          f"{linhas_espacos} mensagens alteradas; {len(df)} linhas antes e depois",
          "Perda de sinais de estilo; pode revelar novas duplicatas")


Mensagens com espaços normalizados: 407
Duplicatas após espaços: 11


## 4. Reparo restrito de pontuação e nova deduplicação
A inspeção encontrou caracteres C1 U+0091, U+0092, U+0093, U+0094 e U+0096 em posições de pontuação. Exemplos: `that\x92s` e `\x93It's ...`. Mapeio somente esses cinco caracteres para suas correspondências de pontuação Windows-1252: aspas curvas e meia-risca. Não tento recodificar o texto inteiro, pois isso poderia danificar símbolos já corretos, como £.

O mapa é manual, não aprende parâmetros. Existe risco de interpretar um caractere legítimo como artefato; o reparo é restrito aos padrões inspecionados. Reconfiro duplicatas após as duas normalizações, pois textos antes diferentes podem ter se tornado iguais. Textos idênticos com rótulos conflitantes interrompem a execução para análise, em vez de receber uma escolha arbitrária.

In [5]:
mapa_pontuacao = {0x91: "‘", 0x92: "’", 0x93: "“", 0x94: "”", 0x96: "–"}
reparado = df["message"].str.translate(mapa_pontuacao)
linhas_pontuacao = int(reparado.ne(df["message"]).sum())
caracteres_pontuacao = sum(sum(ord(c) in mapa_pontuacao for c in t) for t in df["message"])
df["message"] = reparado
print("Mensagens com pontuação reparada:", linhas_pontuacao)
print("Caracteres reparados:", caracteres_pontuacao)
print("U+FFFD restante:", int(df["message"].str.contains("\ufffd", regex=False).sum()))
print("Controles C1 restantes:", int(df["message"].str.contains(r"[\x80-\x9f]", regex=True).sum()))
assert not df["message"].str.contains("\ufffd", regex=False).any()
assert not df["message"].str.contains(r"[\x80-\x9f]", regex=True).any()
registrar("Reparo de cinco caracteres C1", "message", "Corrigir pontuação inspecionada sem recodificar todo o texto",
          f"{linhas_pontuacao} mensagens e {caracteres_pontuacao} caracteres; 0 linhas removidas",
          "Interpretação indevida de caractere; mapa limitado aos casos inspecionados")
conflitos = int(df.groupby("message")["label"].nunique().gt(1).sum())
assert conflitos == 0, "Há textos iguais com rótulos diferentes: revisar manualmente"
antes = len(df)
novas_duplicatas = int(df.duplicated().sum())
classes_removidas = df.loc[df.duplicated(), "label"].value_counts().to_dict()
df = df.drop_duplicates().reset_index(drop=True)
print("Rótulos conflitantes:", conflitos)
print("Nova deduplicação:", antes, "->", len(df))
print("Classes das cópias removidas:", classes_removidas)
registrar("Deduplicação após normalização", "label e message", "Retirar cópias reveladas pela normalização",
          f"{novas_duplicatas} linhas removidas; {antes} -> {len(df)}; classes: {classes_removidas}",
          "Nova alteração das proporções; remoção limitada a pares exatamente iguais")


Mensagens com pontuação reparada: 35
Caracteres reparados: 50
U+FFFD restante: 0
Controles C1 restantes: 0
Rótulos conflitantes: 0
Nova deduplicação: 5171 -> 5159
Classes das cópias removidas: {'spam': 12}


## 5. Codificação do alvo
Uso `ham = 0` e `spam = 1`, sem hierarquia entre classes. Um mapa fixo é suficiente para esse alvo binário, sem criar duas colunas por one-hot encoding. A validação impede que uma classe desconhecida se transforme silenciosamente em nulo. Não foi necessária imputação e não se aplica escala diretamente ao texto.

In [6]:
df["label"] = df["label"].map({"ham": 0, "spam": 1})
assert df["label"].notna().all()
assert set(df["label"].unique()) == {0, 1}
print("Rótulos codificados:", len(df))
print("Tipo do alvo:", df["label"].dtype)
registrar("Codificação binária fixa", "label", "Representar duas classes com um mapa fixo",
          f"{len(df)} valores em 1 coluna; 0 linhas removidas",
          "Categoria desconhecida virar nulo; validação interrompe o processo")
print("Classes finais:\n" + pd.DataFrame({
    "quantidade": df["label"].value_counts(),
    "percentual": df["label"].value_counts(normalize=True).mul(100).round(2)
}).to_string())


Rótulos codificados: 5159
Tipo do alvo: int64
Classes finais:
       quantidade  percentual
label                        
0            4518       87.58
1             641       12.42


## 6. Vazamento de dados e prontidão
**Nenhuma transformação que aprende parâmetros foi ajustada sobre a base inteira antes da divisão treino/teste.** As operações usam regras fixas; não há `fit`, mediana estimada, escala ajustada nem vocabulário aprendido. A divisão treino/teste ainda não foi executada nesta preparação.

As entradas são selecionadas explicitamente como `X = df[["message"]]`; `label` fica somente em `y`. Não existem colunas de identificador ou de informação posterior ao desfecho na fonte selecionada, portanto nenhuma precisou ser removida. O índice não é exportado. Números e URLs dentro da mensagem não são IDs de linha e são preservados como conteúdo textual disponível no recebimento do SMS.

A base está pronta como entrada para o Encontro 9. Na modelagem, dividir com estratificação por `y`; ajustar TF-IDF apenas no treino e aplicar `transform` no teste. A mesma restrição vale para qualquer futura imputação ou escala. Não há garantia de eliminar vazamento por mensagens apenas semelhantes: a conferência cobre igualdade exata.

In [7]:
X = df[["message"]].copy()
y = df["label"].copy()
assert list(X.columns) == ["message"]
assert "label" not in X.columns
assert not df["message"].duplicated().any()
print("Features:", list(X.columns), "| Alvo:", y.name)
print("Identificadores entre as features: 0")
print("Colunas pós-desfecho entre as features: 0")
print("Transformações ajustadas por fit: nenhuma")
print("Etapa seguinte: divisão estratificada, TF-IDF somente no treino e treinamento.")


Features: ['message'] | Alvo: label
Identificadores entre as features: 0
Colunas pós-desfecho entre as features: 0
Transformações ajustadas por fit: nenhuma
Etapa seguinte: divisão estratificada, TF-IDF somente no treino e treinamento.


## 7. Registro de decisões e salvamento verificado
O arquivo `dados_tratados.csv` é a versão final; `registro_decisoes.csv` documenta cada ação. A verificação de leitura compara conteúdo e estrutura, além de dimensões, ausentes e duplicatas. O SHA-256 confere que o ZIP original permaneceu intacto. O índice é descartado apenas como índice de tabela, sem ser exportado como feature.

In [8]:
registro = pd.DataFrame(decisoes)
registro.to_csv("registro_decisoes.csv", index=False)
print(registro.to_string(index=False))
df.to_csv("dados_tratados.csv", index=False)
conferencia = pd.read_csv("dados_tratados.csv")
pd.testing.assert_frame_equal(df, conferencia, check_dtype=False)
assert conferencia.shape == df.shape
assert conferencia.isna().sum().sum() == 0
assert not conferencia.duplicated().any()
assert conferencia["message"].str.strip().ne("").all()
assert hashlib.sha256(origem.read_bytes()).hexdigest() == hash_original
print("Dimensões:", df_original.shape, "->", conferencia.shape)
print("Linhas removidas:", len(df_original) - len(conferencia))
print("Colunas removidas: 0")
print("Ausentes e duplicatas finais: 0 e 0")
print("CSV recarregado e conferido: OK | Original preservado: OK")


                         transformacao  coluna_afetada                                                              motivo                                                   impacto                                                                      risco
Leitura TSV sem interpretação de aspas label e message Preservar cada linha canônica; evitar reconstruir o CSV fragmentado Leitura: 5572 -> 5574 linhas; 2 colunas; original intacto    Mudança futura de formato; comparação linha a linha valida esta leitura
                  Deduplicação inicial label e message                              Evitar peso adicional de cópias exatas   403 linhas removidas; 5574 -> 5171; 0 colunas removidas              Altera frequências; mensagens semelhantes ainda podem existir
               Normalização de espaços         message                Reduzir variações de formatação preservando palavras       407 mensagens alteradas; 5171 linhas antes e depois                   Perda de sinais de estilo; pode r

## 8. O que mudou e como explicar as verificações
- Leitura corrigida: 5.574 registros reais, em vez de 5.572 linhas interpretadas pelo leitor padrão. As aspas são conteúdo de SMS, não delimitadores.
- Novas ações: normalização de espaços e reparo restrito de pontuação; nova deduplicação após normalizar.
- README com público, problema, origem, licença, arquivos e reprodução; decisões também exportadas em CSV.
- A proporção de spam após o esboço era 12,63%; nesta versão, passa a 12,42%. A comparação parte de leituras iniciais diferentes, registradas na seção 1.

**`assert`** verifica uma condição: quando falsa, interrompe o programa com `AssertionError`. Aqui evita aceitar classes inválidas, ausentes ou duplicatas. Use Python sem a opção `-O`, que desativa asserts.

**`pd.testing.assert_frame_equal`** compara dois DataFrames e sinaliza diferenças. Aqui confirma a leitura linha a linha da fonte e que o CSV salvo preserva conteúdo, colunas e ordem. `check_dtype=False` tolera apenas diferenças de representação de tipos entre versões de pandas; não permite valores diferentes.

**IA:** ChatGPT apoiou a revisão do código, as verificações, a execução e a documentação final. O ponto de partida foi o código do projeto fornecido por Lucas. Revisar as decisões e saber explicar os recursos continua sendo responsabilidade do estudante.